Linear Classifier

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.impute import SimpleImputer

import os
import seaborn as sns
import matplotlib.pyplot as plt
from typing import Tuple

In [ ]:
def load_and_prepare_data(benign_file, jailbreak_file, Gride=False,range_scaling=2):
    """
    Loads data from two .npy files, prepares it for scikit-learn, and creates labels.

    Args:
        benign_file (str): Path to the .npy file for the benign class.
        jailbreak_file (str): Path to the .npy file for the jailbreak class.

    Returns:
        tuple: A tuple containing the feature matrix (X) and the label vector (y).
    """
    try:
        # Load the numpy arrays
        benign_data = np.load(benign_file)
        jailbreak_data = np.load(jailbreak_file)
        if Gride:
            benign_data = benign_data[:,:,range_scaling]
            jailbreak_data = jailbreak_data[:,:,range_scaling]
        
        print(f"Loaded '{benign_file}' with shape: {benign_data.shape}")
        print(f"Loaded '{jailbreak_file}' with shape: {jailbreak_data.shape}")

        # Scikit-learn expects data in the format (n_samples, n_features).
        # Our data is (n_features, n_samples), so we need to transpose it.
        benign_samples = benign_data.T
        jailbreak_samples = jailbreak_data.T

        print(f"Transposed benign data shape for training: {benign_samples.shape}")
        print(f"Transposed jailbreak data shape for training: {jailbreak_samples.shape}")

        # Create labels for the data
        # 0 for benign, 1 for jailbreak
        benign_labels = np.zeros(benign_samples.shape[0])
        jailbreak_labels = np.ones(jailbreak_samples.shape[0])

        # Combine the data into a single feature matrix (X) and label vector (y)
        X = np.vstack((benign_samples, jailbreak_samples))
        y = np.concatenate((benign_labels, jailbreak_labels))

                # --- NaN Handling ---
        # Check if any NaN values exist in the combined dataset.
        if np.isnan(X).any():
            print(f"Found {np.isnan(X).sum()} NaN values. Imputing with column mean...")
            
            # Use SimpleImputer to replace NaN with the mean of the feature column.
            # This is a standard practice to handle missing data without losing samples.
            imputer = SimpleImputer(missing_values=np.nan, strategy='mean')
            X = imputer.fit_transform(X)
            
            print("Imputation complete. Dataset is now free of NaN values.")
        else:
            print("No NaN values found in the dataset.")


        return X, y

    except FileNotFoundError as e:
        print(f"Error: {e}. Please make sure the .npy files exist.")
        return None, None

In [ ]:
def train_and_evaluate_classifier(X, y):
    """
    Trains a logistic regression classifier, evaluates with cross-validation,
    and provides a detailed report on a hold-out test set.

    Args:
        X (np.array): The feature matrix.
        y (np.array): The label vector.
    """
    # Initialize the linear classifier
    classifier = LogisticRegression(random_state=42, solver='liblinear')

    # --- 5-Fold Cross-Validation ---
    print("\n--- Cross-Validation Results ---")
    print("Performing 5-fold cross-validation...")
    # Perform cross-validation on the entire dataset
    cv_scores = cross_val_score(classifier, X, y, cv=5, scoring='accuracy')
    print("Cross-validation complete.")
    print(f"  - Accuracy scores for each fold: {cv_scores}")
    print(f"  - Mean accuracy: {cv_scores.mean():.4f}")
    print(f"  - Standard deviation: {cv_scores.std():.4f}")

    # --- Evaluation on a single Hold-Out Test Set ---
    # Split the data into training and testing sets for a detailed report
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    print(f"\nSplitting data for detailed report:")
    print(f"  - Training set size: {X_train.shape[0]} samples")
    print(f"  - Test set size: {X_test.shape[0]} samples")

    # Train the classifier on the 80% training split
    print("\nTraining the classifier on the training set...")
    classifier.fit(X_train, y_train)
    print("Training complete.")

    # Make predictions on the test set
    y_pred = classifier.predict(X_test)
    #plot_logistic_regression_weights(classifier)
    # Evaluate the classifier's performance
    accuracy = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred, target_names=['Benign', 'Jailbreak'])
    conf_matrix = confusion_matrix(y_test, y_pred)

    print("\n--- Detailed Evaluation on Hold-Out Test Set ---")
    print(f"Accuracy on the test set: {accuracy:.4f}")
    print("\nClassification Report:")
    print(report)
    print("Confusion Matrix:")
    print(conf_matrix)
    
    # Visualize the confusion matrix
    plt.figure(figsize=(8, 6))
    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Benign', 'Jailbreak'], 
                yticklabels=['Benign', 'Jailbreak'])
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title('Confusion Matrix on Hold-Out Test Set')
    plt.savefig('confusion_matrix.png')
    print("\nSaved confusion matrix plot to 'confusion_matrix.png'")

def train_and_evaluate_classifier_by_layer(X, y, layer_index):
    """
    Trains and evaluates a classifier using data from only a single specified layer.

    Args:
        X (np.array): The full feature matrix.
        y (np.array): The label vector.
        layer_index (int): The index of the layer to use for classification.
    """
    print(f"\n{'='*25} EVALUATING LAYER {layer_index} {'='*25}")

    # Select data for the specified layer only.
    # We use [:, [layer_index]] to keep it as a 2D array for scikit-learn.
    X_layer = X[:, [layer_index]]

    # Initialize the linear classifier
    classifier = LogisticRegression(random_state=42, solver='liblinear')

    # --- 5-Fold Cross-Validation on the single layer ---
    print(f"\n--- Cross-Validation Results for Layer {layer_index} ---")
    cv_scores = cross_val_score(classifier, X_layer, y, cv=5, scoring='accuracy')
    print(f"  - Mean accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

    # --- Evaluation on a single Hold-Out Test Set for the single layer ---
    X_train, X_test, y_train, y_test = train_test_split(
        X_layer, y, test_size=0.3, random_state=42, stratify=y
    )
    
    # Train the classifier
    classifier.fit(X_train, y_train)

    # Make predictions
    y_pred = classifier.predict(X_test)

    # Evaluate performance
    accuracy = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred, target_names=['Benign', 'Jailbreak'], zero_division=0)
    
    print(f"\n--- Detailed Report on Hold-Out Test Set for Layer {layer_index} ---")
    print(f"Accuracy: {accuracy:.4f}")
    print("Classification Report:")
    print(report)

In [ ]:

benign_file_path = "<path benign file>"   #it has to be a matrix (32,3000)
jailbreak_file_path =  "<path jailbreak file>"    #it has to be a matrix (32,3000)

X_data, y_data = load_and_prepare_data(benign_file_path, jailbreak_file_path,Gride=False,range_scaling=3)


In [ ]:
if X_data is not None and y_data is not None:
        # Train and evaluate the model
    train_and_evaluate_classifier(X_data, y_data)